# 02. 단어 임베딩 (Word Embeddings)

## 학습 목표
- One-hot의 한계를 이해하고 Dense Embedding의 필요성 파악
- Word2Vec Skip-gram을 PyTorch로 직접 구현
- 사전학습 임베딩을 로드하고 시각화

## 핵심 논문
- [Efficient Estimation of Word Representations in Vector Space (Mikolov et al., 2013)](https://arxiv.org/abs/1301.3781)
- [GloVe: Global Vectors for Word Representation (Pennington et al., 2014)](https://nlp.stanford.edu/pubs/glove.pdf)

---

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
import matplotlib.pyplot as plt

## 1. One-hot Encoding의 한계

One-hot은 단어를 **어휘 크기 차원의 희소 벡터**로 표현:

- "king" → [1, 0, 0, 0, 0]
- "queen" → [0, 1, 0, 0, 0]
- "man" → [0, 0, 1, 0, 0]

### 문제점

| 문제 | 설명 |
|------|------|
| 희소성 | 어휘 10만개면 99,999개가 0 → 메모리 낭비 |
| 의미 무시 | king과 queen의 유사성을 표현할 수 없음 |
| 직교성 | 모든 단어 쌍의 코사인 유사도가 0 |

In [ ]:
# One-hot 인코딩의 문제 확인
vocab = ['king', 'queen', 'man', 'woman', 'apple']
V = len(vocab)

# One-hot 벡터 생성
one_hot = {word: np.eye(V)[i] for i, word in enumerate(vocab)}

print("One-hot 인코딩:")
for word, vec in one_hot.items():
    print(f"  {word:6s} → {vec.astype(int)}")

# 코사인 유사도 확인
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(f"\n코사인 유사도:")
print(f"  king vs queen: {cosine_sim(one_hot['king'], one_hot['queen']):.1f}")
print(f"  king vs man:   {cosine_sim(one_hot['king'], one_hot['man']):.1f}")
print(f"  king vs apple: {cosine_sim(one_hot['king'], one_hot['apple']):.1f}")
print(f"\n→ 모든 쌍의 유사도가 0. 의미적 관계를 전혀 반영하지 못함.")

### Dense Embedding의 필요성

**아이디어**: 단어를 저차원(50~300차원)의 밀집 벡터로 표현하면, 비슷한 단어는 비슷한 벡터를 가지게 된다.

$$\text{One-hot: } \mathbb{R}^{|V|} \quad\rightarrow\quad \text{Dense: } \mathbb{R}^d \quad (d \ll |V|)$$

- "king" → [0.23, -0.15, 0.89, ...] (d차원)
- "queen" → [0.25, -0.12, 0.85, ...] (비슷한 벡터!)

> **핵심 직관**: "비슷한 맥락에서 등장하는 단어는 비슷한 의미를 가진다" (Distributional Hypothesis)

---
## 2. Word2Vec: Skip-gram

### 핵심 아이디어

중심 단어(center word)가 주어졌을 때, 주변 단어(context words)를 예측하는 방식으로 학습.

문장: "The **cat** sat on the mat"
- 중심어: "cat", 윈도우 크기 2
- 학습 쌍: (cat, The), (cat, sat), (cat, on)

### 수식

중심 단어 $w_c$가 주어졌을 때 주변 단어 $w_o$의 확률:

$$P(w_o | w_c) = \frac{\exp(\mathbf{u}_{w_o}^\top \mathbf{v}_{w_c})}{\sum_{w=1}^{|V|} \exp(\mathbf{u}_w^\top \mathbf{v}_{w_c})}$$

- $\mathbf{v}_{w_c}$: 중심 단어의 임베딩 벡터 (입력 행렬)
- $\mathbf{u}_{w_o}$: 주변 단어의 임베딩 벡터 (출력 행렬)
- 분모: 전체 어휘에 대한 softmax (계산이 매우 비쌈)

### 구조

```
입력 (one-hot)    임베딩 행렬 W      출력 행렬 W'
[0,0,1,0,0]  →   [___d___]     →   softmax → [p1, p2, p3, p4, p5]
  (|V|)           (d dim)             (|V|)
```

### 2.1 학습 데이터 준비

In [ ]:
# 간단한 코퍼스
corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "the cat chased the dog",
    "the dog chased the cat",
    "the mat is on the floor",
    "the cat is on the mat",
    "the dog is on the floor",
]

# 토큰화 및 어휘 구축
tokenized = [sent.split() for sent in corpus]
all_words = [w for sent in tokenized for w in sent]
vocab = sorted(set(all_words))
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
V = len(vocab)

print(f"어휘: {vocab}")
print(f"어휘 크기: {V}")
print(f"전체 토큰 수: {len(all_words)}")

In [ ]:
# Skip-gram 학습 쌍 생성
def create_skipgram_pairs(tokenized_corpus, window_size=2):
    """(center_word, context_word) 쌍 생성"""
    pairs = []
    for sentence in tokenized_corpus:
        for i, center in enumerate(sentence):
            # 윈도우 내 주변 단어
            for j in range(max(0, i - window_size), min(len(sentence), i + window_size + 1)):
                if i != j:
                    pairs.append((word2idx[center], word2idx[sentence[j]]))
    return pairs

pairs = create_skipgram_pairs(tokenized, window_size=2)
print(f"총 학습 쌍: {len(pairs)}개")
print(f"\n처음 10개 예시:")
for center_idx, context_idx in pairs[:10]:
    print(f"  ({idx2word[center_idx]:6s}, {idx2word[context_idx]})")

### 2.2 Skip-gram 모델 구현 (PyTorch)

In [ ]:
class SkipGram(nn.Module):
    """Skip-gram 모델 (Softmax 버전)"""
    
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        # 중심 단어 임베딩 (입력)
        self.center_embeddings = nn.Embedding(vocab_size, embedding_dim)
        # 주변 단어 임베딩 (출력)
        self.context_embeddings = nn.Embedding(vocab_size, embedding_dim)
    
    def forward(self, center_ids, context_ids):
        # 중심 단어 벡터: (batch, dim)
        center_vecs = self.center_embeddings(center_ids)
        # 주변 단어 벡터: (batch, dim)
        context_vecs = self.context_embeddings(context_ids)
        # 내적: 두 벡터가 비슷할수록 높은 점수
        scores = torch.sum(center_vecs * context_vecs, dim=1)
        return scores

# 하이퍼파라미터
EMBEDDING_DIM = 10
LEARNING_RATE = 0.01
EPOCHS = 100

model = SkipGram(V, EMBEDDING_DIM)
print(f"모델 구조:\n{model}")
print(f"\n파라미터 수: {sum(p.numel() for p in model.parameters())}")
print(f"  = 2 x {V} x {EMBEDDING_DIM} (center + context 임베딩)")

---
## 3. Negative Sampling

### 왜 필요한가?

Softmax의 분모에서 **전체 어휘에 대한 합**을 계산해야 함:

$$P(w_o | w_c) = \frac{\exp(\mathbf{u}_{w_o}^\top \mathbf{v}_{w_c})}{\sum_{w=1}^{\mathbf{|V|}} \exp(\mathbf{u}_w^\top \mathbf{v}_{w_c})}$$

- 어휘가 10만개면 매 학습마다 10만번 계산 → **매우 느림**

### Negative Sampling 아이디어

전체 어휘 대신 **소수의 부정 샘플(negative samples)**만 사용:

- 긍정 쌍: (cat, sat) → 실제 함께 등장한 쌍 → 점수를 높임
- 부정 쌍: (cat, floor), (cat, log) → 랜덤 선택 → 점수를 낮춤

$$\mathcal{L} = -\log\sigma(\mathbf{u}_{w_o}^\top \mathbf{v}_{w_c}) - \sum_{k=1}^{K} \log\sigma(-\mathbf{u}_{w_k}^\top \mathbf{v}_{w_c})$$

- $\sigma$: sigmoid 함수
- $K$: 부정 샘플 수 (보통 5~20)

In [ ]:
class SkipGramNegSampling(nn.Module):
    """Skip-gram with Negative Sampling"""
    
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.center_embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.context_embeddings = nn.Embedding(vocab_size, embedding_dim)
        
        # 초기화: 작은 값으로
        nn.init.uniform_(self.center_embeddings.weight, -0.5/embedding_dim, 0.5/embedding_dim)
        nn.init.zeros_(self.context_embeddings.weight)
    
    def forward(self, center_ids, pos_ids, neg_ids):
        """
        center_ids: (batch,)
        pos_ids: (batch,) 긍정 샘플
        neg_ids: (batch, K) 부정 샘플
        """
        center_vecs = self.center_embeddings(center_ids)    # (batch, dim)
        pos_vecs = self.context_embeddings(pos_ids)         # (batch, dim)
        neg_vecs = self.context_embeddings(neg_ids)         # (batch, K, dim)
        
        # 긍정 쌍 점수: 높아야 함
        pos_score = torch.sum(center_vecs * pos_vecs, dim=1)  # (batch,)
        pos_loss = -torch.log(torch.sigmoid(pos_score) + 1e-10)
        
        # 부정 쌍 점수: 낮아야 함
        neg_score = torch.bmm(neg_vecs, center_vecs.unsqueeze(2)).squeeze(2)  # (batch, K)
        neg_loss = -torch.log(torch.sigmoid(-neg_score) + 1e-10).sum(dim=1)
        
        return (pos_loss + neg_loss).mean()


# 부정 샘플링 함수
def get_negative_samples(batch_size, num_neg, vocab_size, word_freq):
    """빈도 기반 부정 샘플링 (빈도^0.75 확률)"""
    weights = np.array([word_freq.get(i, 1) for i in range(vocab_size)]) ** 0.75
    weights /= weights.sum()
    neg_samples = np.random.choice(vocab_size, size=(batch_size, num_neg), p=weights)
    return torch.LongTensor(neg_samples)


# 단어 빈도 계산
word_freq = Counter(word2idx[w] for w in all_words)
print(f"단어 빈도: {dict(sorted(word_freq.items(), key=lambda x: -x[1])[:5])}")
print(f"(인덱스: 빈도) → 'the'({word2idx['the']})이 가장 빈번")

In [ ]:
# 학습
NUM_NEG = 5
EMBEDDING_DIM = 10
EPOCHS = 200
LEARNING_RATE = 0.025

model = SkipGramNegSampling(V, EMBEDDING_DIM)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 학습 데이터 텐서 변환
center_ids = torch.LongTensor([p[0] for p in pairs])
context_ids = torch.LongTensor([p[1] for p in pairs])

losses = []
for epoch in range(EPOCHS):
    # 부정 샘플 생성
    neg_ids = get_negative_samples(len(pairs), NUM_NEG, V, word_freq)
    
    # Forward
    loss = model(center_ids, context_ids, neg_ids)
    
    # Backward
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {loss.item():.4f}")

# 학습 곡선
plt.figure(figsize=(8, 3))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Skip-gram with Negative Sampling: Training Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 학습된 임베딩 확인
embeddings = model.center_embeddings.weight.detach().numpy()

# 단어 간 유사도 확인
def word_similarity(word1, word2, embeddings):
    v1 = embeddings[word2idx[word1]]
    v2 = embeddings[word2idx[word2]]
    return cosine_sim(v1, v2)

print("학습된 임베딩 기반 유사도:")
pairs_to_check = [
    ('cat', 'dog'),   # 비슷한 맥락에 등장
    ('mat', 'floor'), # 비슷한 맥락에 등장
    ('cat', 'floor'), # 다른 맥락
    ('sat', 'chased'),# 둘 다 동사
]

for w1, w2 in pairs_to_check:
    sim = word_similarity(w1, w2, embeddings)
    print(f"  {w1:7s} vs {w2:7s}: {sim:.4f}")

---
## 4. GloVe (Global Vectors for Word Representation)

### Word2Vec vs GloVe

| | Word2Vec | GloVe |
|---|---|---|
| 방식 | 로컬 윈도우 (예측 기반) | 글로벌 공출현 행렬 (카운트 기반) |
| 학습 목표 | P(context|center) 예측 | 공출현 확률 비율 복원 |
| 장점 | 학습 효율적 | 전체 통계 활용 |

### GloVe의 핵심 아이디어

공출현 행렬 $X$에서 단어 간 관계를 포착:

$$\mathbf{w}_i^\top \mathbf{w}_j + b_i + b_j = \log X_{ij}$$

→ 공출현 빈도의 로그값을 두 벡터의 내적으로 근사.

### 사전학습 GloVe 로드

In [ ]:
# 사전학습 GloVe 임베딩 로드 (gensim 사용)
# Colab에서 실행 시:
# !pip install gensim

# 방법 1: gensim의 사전학습 모델 (다운로드 필요)
# import gensim.downloader as api
# glove = api.load('glove-wiki-gigaword-50')  # 50차원 GloVe

# 방법 2: 직접 GloVe 파일 로드 (가볍게 시연하기 위해 가상 임베딩 사용)
# 실제로는 https://nlp.stanford.edu/projects/glove/ 에서 다운로드

# 시연용 가상 사전학습 임베딩 (실제 GloVe의 관계를 모사)
np.random.seed(42)
pretrained_dim = 50

# 의미적 관계를 가지도록 수동 설정
base = np.random.randn(pretrained_dim)
royalty = np.random.randn(pretrained_dim) * 0.5
gender = np.random.randn(pretrained_dim) * 0.5
animal = np.random.randn(pretrained_dim) * 0.5

pretrained = {
    'king':   base + royalty + gender * 0.3,
    'queen':  base + royalty - gender * 0.3,
    'man':    base + gender * 0.3,
    'woman':  base - gender * 0.3,
    'prince': base + royalty * 0.7 + gender * 0.3,
    'princess': base + royalty * 0.7 - gender * 0.3,
    'dog':    animal + np.random.randn(pretrained_dim) * 0.1,
    'cat':    animal + np.random.randn(pretrained_dim) * 0.1,
    'puppy':  animal + np.random.randn(pretrained_dim) * 0.15,
    'kitten': animal + np.random.randn(pretrained_dim) * 0.15,
    'car':    np.random.randn(pretrained_dim),
    'bus':    np.random.randn(pretrained_dim) * 0.8,
    'train':  np.random.randn(pretrained_dim) * 0.8,
}

# 유사도 확인
print("사전학습 임베딩 유사도:")
test_pairs = [
    ('king', 'queen'),
    ('man', 'woman'),
    ('dog', 'cat'),
    ('king', 'dog'),
    ('king', 'car'),
]

for w1, w2 in test_pairs:
    sim = cosine_sim(pretrained[w1], pretrained[w2])
    print(f"  {w1:8s} vs {w2:8s}: {sim:.4f}")

---
## 5. Embedding 시각화: t-SNE

고차원(50~300차원) 임베딩을 2D로 투영하여 시각화.

**t-SNE (t-distributed Stochastic Neighbor Embedding)**:
- 고차원에서 가까운 점은 2D에서도 가깝게 유지
- 비선형 차원 축소 → PCA보다 클러스터를 잘 보여줌
- 주의: 축의 크기에 의미 없음, 실행마다 결과 다름

In [ ]:
from sklearn.manifold import TSNE

# 임베딩 행렬 구성
words = list(pretrained.keys())
vectors = np.array([pretrained[w] for w in words])

# t-SNE로 2D 투영
tsne = TSNE(n_components=2, random_state=42, perplexity=4)
vectors_2d = tsne.fit_transform(vectors)

# 시각화
fig, ax = plt.subplots(figsize=(10, 8))

# 카테고리별 색상
categories = {
    'royalty': ['king', 'queen', 'prince', 'princess'],
    'human':   ['man', 'woman'],
    'animal':  ['dog', 'cat', 'puppy', 'kitten'],
    'vehicle': ['car', 'bus', 'train'],
}
colors = {'royalty': 'red', 'human': 'blue', 'animal': 'green', 'vehicle': 'orange'}

for cat, cat_words in categories.items():
    idxs = [words.index(w) for w in cat_words]
    ax.scatter(vectors_2d[idxs, 0], vectors_2d[idxs, 1],
              c=colors[cat], s=100, label=cat, zorder=2)

# 단어 레이블
for i, word in enumerate(words):
    ax.annotate(word, (vectors_2d[i, 0] + 0.5, vectors_2d[i, 1] + 0.5), fontsize=11)

ax.set_title('Word Embeddings (t-SNE 2D projection)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("→ 비슷한 의미의 단어들이 클러스터를 이루는 것을 확인")

---
## 6. 아날로지 실험

Word Embedding의 가장 유명한 성질:

$$\vec{king} - \vec{man} + \vec{woman} \approx \vec{queen}$$

임베딩 공간에서 **의미적 관계가 벡터 산술로 표현**된다.

- "king - man" = 왕족성(royalty)
- "woman + 왕족성" = queen

In [ ]:
def analogy(a, b, c, embeddings, topn=3):
    """
    a:b = c:? 관계를 풀어줌
    result = b - a + c (= a에서 b로의 관계를 c에 적용)
    """
    target = embeddings[b] - embeddings[a] + embeddings[c]
    
    # 가장 유사한 단어 찾기 (a, b, c 제외)
    similarities = {}
    for word, vec in embeddings.items():
        if word not in [a, b, c]:
            similarities[word] = cosine_sim(target, vec)
    
    ranked = sorted(similarities.items(), key=lambda x: -x[1])
    return ranked[:topn]

# 아날로지 테스트
print("=== 아날로지 실험 ===")
print()

# king - man + woman = ?
result = analogy('man', 'king', 'woman', pretrained)
print(f"man : king = woman : ?")
print(f"  → king - man + woman ≈ {result[0][0]} ({result[0][1]:.4f})")
for word, sim in result:
    print(f"     {word}: {sim:.4f}")
print()

# king - man + woman = ? (다른 방향)
result = analogy('king', 'queen', 'prince', pretrained)
print(f"king : queen = prince : ?")
print(f"  → queen - king + prince ≈ {result[0][0]} ({result[0][1]:.4f})")
for word, sim in result:
    print(f"     {word}: {sim:.4f}")
print()

# dog - puppy + kitten = ?
result = analogy('puppy', 'dog', 'kitten', pretrained)
print(f"puppy : dog = kitten : ?")
print(f"  → dog - puppy + kitten ≈ {result[0][0]} ({result[0][1]:.4f})")
for word, sim in result:
    print(f"     {word}: {sim:.4f}")

In [ ]:
# 아날로지를 시각적으로 확인
fig, ax = plt.subplots(figsize=(8, 6))

# 관련 단어만 추출하여 시각화
analogy_words = ['king', 'queen', 'man', 'woman']
analogy_vecs = np.array([pretrained[w] for w in analogy_words])

# PCA로 2D 투영 (아날로지 관계를 보기에 더 적합)
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
vecs_2d = pca.fit_transform(analogy_vecs)

# 점 그리기
for i, word in enumerate(analogy_words):
    ax.scatter(vecs_2d[i, 0], vecs_2d[i, 1], s=150, zorder=3)
    ax.annotate(word, (vecs_2d[i, 0] + 0.02, vecs_2d[i, 1] + 0.02), fontsize=14, fontweight='bold')

# 관계 화살표
# man → king (왕족성)
ax.annotate('', xy=vecs_2d[0], xytext=vecs_2d[2],
           arrowprops=dict(arrowstyle='->', color='red', lw=2))
# woman → queen (왕족성)
ax.annotate('', xy=vecs_2d[1], xytext=vecs_2d[3],
           arrowprops=dict(arrowstyle='->', color='red', lw=2, linestyle='--'))
# man → woman (성별)
ax.annotate('', xy=vecs_2d[3], xytext=vecs_2d[2],
           arrowprops=dict(arrowstyle='->', color='blue', lw=2))
# king → queen (성별)
ax.annotate('', xy=vecs_2d[1], xytext=vecs_2d[0],
           arrowprops=dict(arrowstyle='->', color='blue', lw=2, linestyle='--'))

ax.set_title('Word Analogy: king - man + woman = queen\n'
            'Red: royalty direction, Blue: gender direction')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 연습 문제

아래 문제를 직접 풀어보세요.

### 연습 1: CBOW 모델 구현

Skip-gram의 반대 방향인 CBOW(Continuous Bag of Words)를 구현하세요.
- Skip-gram: 중심 → 주변 예측
- CBOW: 주변 → 중심 예측

In [ ]:
# TODO: CBOW 모델을 구현하세요
#
# 1. CBOW 학습 데이터 생성 함수:
#    - 입력: 주변 단어들의 인덱스 리스트
#    - 출력: 중심 단어 인덱스
#    예: [the, sat, on, the] → cat
#
# 2. CBOW 모델 클래스:
#    - 주변 단어 임베딩의 평균을 중심 단어 예측에 사용
#    - forward(context_ids) → center_word 확률
#
# 3. 학습 후 Skip-gram과 유사도 결과를 비교하세요
#
# 힌트: context_vecs.mean(dim=1)로 주변 단어 평균 계산


### 연습 2: 사전학습 GloVe 로드 및 탐색

gensim을 사용하여 실제 GloVe 임베딩을 로드하고 탐색하세요.

In [ ]:
# TODO: 실제 GloVe 임베딩으로 실험 (Colab에서 실행 권장)
#
# import gensim.downloader as api
# glove = api.load('glove-wiki-gigaword-50')  # 약 65MB
#
# 1. 'python'과 가장 유사한 단어 10개를 찾으세요
#    glove.most_similar('python', topn=10)
#
# 2. 아날로지 실험:
#    - paris : france = tokyo : ?  (나라-수도 관계)
#    - slow : slower = fast : ?    (비교급 관계)
#    glove.most_similar(positive=['france', 'tokyo'], negative=['paris'])
#
# 3. t-SNE로 프로그래밍 언어 관련 단어 20개를 시각화하세요
#    예: python, java, javascript, c, ruby, ...


---
## 핵심 정리

| 개념 | 설명 | ML에서의 역할 |
|------|------|---------------|
| One-hot | 희소 벡터, 의미 무시 | 기본 인코딩이지만 한계 큼 |
| Dense Embedding | 저차원 밀집 벡터 | 의미적 유사성 반영 |
| Word2Vec (Skip-gram) | 중심→주변 예측으로 학습 | 빠르고 효과적인 임베딩 학습 |
| Negative Sampling | softmax 대신 소수 부정 샘플 | 학습 속도 대폭 개선 |
| GloVe | 공출현 통계 기반 | 전체 코퍼스 통계 활용 |
| t-SNE | 고차원 → 2D 시각화 | 임베딩 품질 확인 |
| 아날로지 | king - man + woman = queen | 벡터 연산으로 의미 관계 표현 |

**다음 노트북**: [03-text-similarity.ipynb](03-text-similarity.ipynb) - 텍스트 유사도